<a href="https://colab.research.google.com/github/DDarion/compressed-gaussians-colab/blob/main/ECNG_3020_3DGS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Set-up
## Confirming the GPU being used

In [ ]:
!nvidia-smi

Fri Nov 13 16:28:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Version comparions

In [ ]:
# nvcc and pytorch cuda version
%cd /content/gaussian-splatting
!nvcc --version
import torch
print(torch.__version__, torch.version.cuda)
!pip install -v --no-build-isolation submodules/diff-gaussian-rasterization 2>&1 | tail -60

[Errno 2] No such file or directory: '/content/gaussian-splatting'
/content
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
2.14.1+cu130 13.0
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
ERROR: Invalid requirement: 'submodules/diff-gaussian-rasterization': Expected semicolon (after name with no version specifier) or end
    submodules/diff-gaussian-rasterization
              ^
Hint: It looks like a path. File 'submodules/diff-gaussian-rasterization' does not exist.


# Setting up baseline Kerbl model
## Splatting

In [1]:
# mounting Drive first so outputs survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
RUN = "/content/drive/MyDrive/3dgs_outputs/train_baseline"   # where the output goes

# getting the code from kerbl repo
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
!pip install -q plyfile

# patching and building the CUDA extensions
%cd /content/gaussian-splatting
!grep -q "<cstdint>" submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '0,/#pragma once/s//#pragma once\n#include <cstdint>/' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!head -15 submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!pip install --no-build-isolation submodules/diff-gaussian-rasterization
!pip install --no-build-isolation submodules/simple-knn
!python -c "import diff_gaussian_rasterization, simple_knn; print('ok')"

# getting the dataset
%cd /content/gaussian-splatting
# below is the main dataset
!wget -nc -q https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
# below is the mirror dataset
# !wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip -oq tandt_db.zip
!date # to show the date when the model was trained
!ls tandt

# training the model (--eval holds out test views, -m saves into Drive)
!python train.py -s /content/gaussian-splatting/tandt/train --eval -m {RUN}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.72 MiB | 25.99 MiB/s, done.
Resolving deltas: 100% (593/593), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remo